
# ─────────────────────────────────────────────
# TITLE & OVERVIEW
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("""# 🌍 Synthetic Seismic Data Generation — Full Professional Notebook
## From Physical Theory to Geological Interpretation

---

**Author:** Seismic Geophysics Lab  
**Purpose:** Generate realistic 2-D synthetic seismic sections containing faults, fractures, folds, unconformities, salt domes, and diffractions — from first principles.

---

## 📚 Table of Contents

1. [Theoretical Background](#1-theoretical-background)  
2. [Setup & Imports](#2-setup--imports)  
3. [Earth Model Builder](#3-earth-model-builder)  
4. [Geological Feature Generators](#4-geological-feature-generators)  
    - 4a. Horizontal Layering  
    - 4b. Folds (Anticline / Syncline)  
    - 4c. Normal & Reverse Faults  
    - 4d. Fracture Zones  
    - 4e. Unconformities  
    - 4f. Salt Dome  
    - 4g. Pinch-outs & Stratigraphic Traps  
5. [Wavelet Design & Convolution Model](#5-wavelet-design--convolution-model)  
6. [Full Synthetic Seismic Section](#6-full-synthetic-seismic-section)  
7. [Seismic Attributes](#7-seismic-attributes)  
8. [Noise & Acquisition Effects](#8-noise--acquisition-effects)  
9. [Composite Panel — Publication-Quality Figure](#9-composite-panel--publication-quality-figure)  
10. [Interpretation Overlay](#10-interpretation-overlay)  

---
"""))

# ─────────────────────────────────────────────
# 1. THEORY
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("""## 1  Theoretical Background

### 1.1  The Reflection Seismic Method

Seismic reflection surveys record elastic waves that travel downward through the Earth,
reflect at acoustic-impedance contrasts, and return to surface receivers. The primary
physical quantity is **acoustic impedance** $Z$:

$$Z_i = \\rho_i \\, V_{p,i}$$

where $\\rho_i$ is bulk density $[\\text{kg m}^{-3}]$ and $V_{p,i}$ is P-wave velocity $[\\text{m s}^{-1}]$.

### 1.2  Reflection Coefficient

At a planar interface between layers $i$ and $i{+}1$ (normal incidence):

$$R_i = \\frac{Z_{i+1} - Z_i}{Z_{i+1} + Z_i}$$

$R_i \\in [-1,+1]$; positive $R$ = harder rock below.

### 1.3  The Convolutional Model

The recorded seismic trace $s(t)$ is the convolution of the source wavelet $w(t)$
with the **reflectivity series** $r(t)$ plus noise $n(t)$:

$$s(t) = w(t) * r(t) + n(t)$$

This is the foundation of all synthetic seismogram generation.

### 1.4  Ricker (Mexican-Hat) Wavelet

The most widely used zero-phase source wavelet:

$$w(t) = \\left(1 - 2\\pi^2 f_0^2 t^2\\right)\\, e^{-\\pi^2 f_0^2 t^2}$$

with dominant (peak) frequency $f_0$ [Hz].  
Its Fourier amplitude spectrum is:

$$W(f) = \\frac{2}{\\sqrt{\\pi}}\\,\\frac{f^2}{f_0^3}\\,e^{-(f/f_0)^2}$$

### 1.5  Ormsby Bandpass Wavelet

A trapezoidal-spectrum wavelet defined by four corner frequencies
$f_1 < f_2 < f_3 < f_4$ (all in Hz):

$$W(f) = \\begin{cases}
0 & f < f_1 \\\\
\\text{ramp up} & f_1 \\le f < f_2 \\\\
1 & f_2 \\le f \\le f_3 \\\\
\\text{ramp down} & f_3 < f \\le f_4 \\\\
0 & f > f_4
\\end{cases}$$

### 1.6  Geological Structures & Their Seismic Expression

| Structure | Physical Cause | Seismic Signature |
|-----------|----------------|-------------------|
| **Horizontal layers** | Sedimentary deposition | Parallel, continuous reflections |
| **Anticline / Syncline** | Compressive folding | Curved, arched / bowl-shaped reflections |
| **Normal fault** | Extensional tectonics | Offset reflections, fault-plane shadow |
| **Reverse / thrust fault** | Compressional tectonics | Doubled reflections, high dip |
| **Fracture zone** | Shear / tension | Chaotic zone, amplitude dimming |
| **Unconformity** | Erosion + re-deposition | Truncation of underlying reflections |
| **Salt dome** | Density inversion (halite) | Pull-up / push-down, edge diffraction |
| **Pinch-out** | Lateral facies change | Reflection termination |
| **Diffraction** | Point scatterer | Hyperbolic event |

### 1.7  Seismic Attributes

Trace-based attributes extracted from the analytic signal $A(t)$:

$$A(t) = s(t) + i\\,\\mathcal{H}\\{s(t)\\} = E(t)\\,e^{i\\phi(t)}$$

- **Instantaneous amplitude (envelope):** $E(t) = |A(t)|$  
- **Instantaneous phase:** $\\phi(t) = \\arg A(t)$  
- **Instantaneous frequency:** $f_i(t) = \\frac{1}{2\\pi}\\frac{d\\phi}{dt}$  
- **Cosine of instantaneous phase:** $\\cos\\phi(t)$ — highlights structural continuity  

---
"""))


In [ ]:
"""
Script to generate the synthetic seismic Jupyter notebook via nbformat.
"""
import nbformat
from nbformat.v4 import new_notebook, new_markdown_cell, new_code_cell

cells = []

# ─────────────────────────────────────────────
# 2. SETUP & IMPORTS
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("## 2  Setup & Imports"))
cells.append(new_code_cell("""import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MultipleLocator
from scipy.signal import hilbert, butter, filtfilt, convolve
from scipy.ndimage import gaussian_filter, uniform_filter
from scipy.interpolate import interp1d
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
np.random.seed(42)

# ── Global figure style ───────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 120,
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

print("✅  All packages loaded successfully.")
"""))

# ─────────────────────────────────────────────
# 3. EARTH MODEL
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("""## 3  Earth Model Builder

We build a 2-D grid of velocity $V_p$ and density $\\rho$, then derive impedance $Z = \\rho V_p$
and the reflectivity cube $R$.
"""))
cells.append(new_code_cell("""# ═══════════════════════════════════════════════════════════
#  GRID PARAMETERS
# ═══════════════════════════════════════════════════════════
NZ   = 500          # samples in depth / time  (vertical)
NX   = 300          # traces (horizontal)
DZ   = 4.0          # sample interval [m]  (depth domain)
DX   = 10.0         # trace spacing [m]
DT   = 0.002        # time sampling [s]  (for convolution)

# Depth axis (m) and trace axis (m)
depth = np.arange(NZ) * DZ           # 0 … 1996 m
x_pos = np.arange(NX) * DX           # 0 … 2990 m

# ── Reference lithologies ─────────────────────────────────────
#  (Vp [m/s], density [kg/m³], description)
LITHO = {
    'shale'       : (2400,  2300, 'Shale'),
    'sandstone'   : (3100,  2400, 'Sandstone'),
    'limestone'   : (4200,  2550, 'Limestone'),
    'dolomite'    : (5000,  2700, 'Dolomite'),
    'coal'        : (1800,  1400, 'Coal'),
    'salt'        : (4480,  2160, 'Halite / Salt'),
    'anhydrite'   : (5500,  2960, 'Anhydrite'),
    'basement'    : (5800,  2850, 'Crystalline Basement'),
    'fracture_zone': (2000, 2100, 'Fracture Zone'),
    'gas_sand'    : (2600,  2200, 'Gas-bearing Sand'),
    'brine_sand'  : (3400,  2450, 'Brine-bearing Sand'),
}

# ── Initialise Vp & density arrays ───────────────────────────
Vp  = np.full((NZ, NX), LITHO['shale'][0], dtype=float)
Rho = np.full((NZ, NX), LITHO['shale'][1], dtype=float)

print(f"Grid: {NZ} samples × {NX} traces")
print(f"Depth range: 0 – {depth[-1]:.0f} m")
print(f"Lateral range: 0 – {x_pos[-1]:.0f} m")
"""))

# ─────────────────────────────────────────────
# 4. GEOLOGICAL FEATURES
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("## 4  Geological Feature Generators"))

cells.append(new_markdown_cell("### 4a  Horizontal & Gently Dipping Layers"))
cells.append(new_code_cell("""def add_layer(Vp, Rho, z_top_func, z_bot_func, vp_val, rho_val):
    \"\"\"
    Fill cells between z_top_func(x) and z_bot_func(x) with given Vp / Rho.
    Both functions accept an array of x-indices and return depth-sample indices.
    \"\"\"
    NZ, NX = Vp.shape
    for ix in range(NX):
        zt = int(np.clip(z_top_func(ix), 0, NZ-1))
        zb = int(np.clip(z_bot_func(ix), 0, NZ-1))
        if zt >= zb:
            continue
        Vp [zt:zb, ix] = vp_val
        Rho[zt:zb, ix] = rho_val

# ── Sedimentary sequence ──────────────────────────────────────
#  Layer 0: Near-surface shale  0–120 m  (already default)
#  Layer 1: Shallow sandstone   120–200 m
add_layer(Vp, Rho,
          lambda ix: np.full_like(ix, 30, dtype=float) if hasattr(ix,'__len__') else 30,
          lambda ix: np.full_like(ix, 50, dtype=float) if hasattr(ix,'__len__') else 50,
          *LITHO['sandstone'][:2])

# Use scalar version for clarity
for ix in range(NX):
    # Layer 1: sandstone 120–200 m
    Vp [30:50,  ix] = LITHO['sandstone'][0]
    Rho[30:50,  ix] = LITHO['sandstone'][1]
    # Layer 2: shale 200–280 m
    Vp [50:70,  ix] = LITHO['shale'][0]
    Rho[50:70,  ix] = LITHO['shale'][1]
    # Layer 3: limestone 280–440 m
    Vp [70:110, ix] = LITHO['limestone'][0]
    Rho[70:110, ix] = LITHO['limestone'][1]
    # Layer 4: shale 440–520 m
    Vp[110:130, ix] = LITHO['shale'][0]
    Rho[110:130,ix] = LITHO['shale'][1]
    # Layer 5: coal seam 520–540 m
    Vp[130:135, ix] = LITHO['coal'][0]
    Rho[130:135,ix] = LITHO['coal'][1]
    # Layer 6: sandstone 540–680 m
    Vp[135:170, ix] = LITHO['sandstone'][0]
    Rho[135:170,ix] = LITHO['sandstone'][1]
    # Layer 7: shale 680–760 m
    Vp[170:190, ix] = LITHO['shale'][0]
    Rho[170:190,ix] = LITHO['shale'][1]
    # Layer 8: dolomite 760–920 m
    Vp[190:230, ix] = LITHO['dolomite'][0]
    Rho[190:230,ix] = LITHO['dolomite'][1]
    # Layer 9: shale 920–1040 m
    Vp[230:260, ix] = LITHO['shale'][0]
    Rho[230:260,ix] = LITHO['shale'][1]
    # Layer 10: limestone 1040–1200 m
    Vp[260:300, ix] = LITHO['limestone'][0]
    Rho[260:300,ix] = LITHO['limestone'][1]
    # Layer 11: deep shale 1200–1600 m
    Vp[300:400, ix] = LITHO['shale'][0]+200   # slight compaction
    Rho[300:400,ix] = LITHO['shale'][1]+50
    # Layer 12: basement >1600 m
    Vp[400:,    ix] = LITHO['basement'][0]
    Rho[400:,   ix] = LITHO['basement'][1]

print("✅  Background stratigraphy added.")
"""))

cells.append(new_markdown_cell("""### 4b  Folds — Anticline & Syncline

An **anticline** is produced by upward arching of layers; a **syncline** by downward bowing.
Mathematically we displace each reflector by a sinusoidal perturbation:

$$z'(x) = z_0 + A \\sin\\!\\left(\\frac{2\\pi(x-x_c)}{\\lambda}\\right)$$

where $A$ is amplitude [samples], $\\lambda$ wavelength, $x_c$ centre position.
"""))
cells.append(new_code_cell("""def apply_fold(Vp, Rho, x_centre_frac, amplitude, wavelength_frac,
               fold_type='anticline'):
    \"\"\"
    Fold the model by vertically displacing samples using a Gaussian-modulated cosine.
    fold_type: 'anticline' (arch up) | 'syncline' (bowl down)
    \"\"\"
    NZ, NX = Vp.shape
    xc     = int(x_centre_frac * NX)
    lam    = int(wavelength_frac * NX)
    sign   = -1 if fold_type == 'anticline' else +1

    Vp_new  = Vp .copy()
    Rho_new = Rho.copy()

    for ix in range(NX):
        # Gaussian envelope × cosine
        envelope = np.exp(-0.5*((ix-xc)/(lam/4))**2)
        shift    = int(sign * amplitude * envelope)
        if shift == 0:
            continue
        col_vp  = Vp [:, ix].copy()
        col_rho = Rho[:, ix].copy()
        # Roll (cyclic shift) the column
        Vp_new [:, ix] = np.roll(col_vp,  shift)
        Rho_new[:, ix] = np.roll(col_rho, shift)
        # Fill the gap at the edge with the edge value
        if shift > 0:
            Vp_new [:shift,  ix] = col_vp [0]
            Rho_new[:shift,  ix] = col_rho[0]
        else:
            Vp_new [shift:,  ix] = col_vp [-1]
            Rho_new[shift:,  ix] = col_rho[-1]

    return Vp_new, Rho_new

# ── Anticline centred at x=40% of section, amplitude 25 samples ──
Vp, Rho = apply_fold(Vp, Rho, x_centre_frac=0.38, amplitude=25,
                     wavelength_frac=0.35, fold_type='anticline')

# ── Syncline centred at x=72% ──────────────────────────────────
Vp, Rho = apply_fold(Vp, Rho, x_centre_frac=0.72, amplitude=15,
                     wavelength_frac=0.25, fold_type='syncline')

print("✅  Anticline (x≈38%) and syncline (x≈72%) added.")
"""))

cells.append(new_markdown_cell("""### 4c  Faults — Normal & Reverse

A **normal fault** (extensional) throws the hanging-wall block downward.
A **reverse / thrust fault** (compressional) throws it upward.

The fault throw $T$ varies with depth — maximum in the middle, tapering to zero at
surface and at depth:

$$T(z) = T_{\\max} \\sin\\!\\left(\\frac{\\pi(z-z_1)}{z_2-z_1}\\right)$$
"""))
cells.append(new_code_cell("""def apply_fault(Vp, Rho, x_fault_frac, dip_deg, throw_max,
                z_top_frac, z_bot_frac, fault_type='normal'):
    \"\"\"
    Apply a planar fault with variable throw.
    x_fault_frac : surface intercept as fraction of NX
    dip_deg      : dip angle from vertical (0 = vertical)
    throw_max    : maximum vertical throw [samples]
    z_top/bot    : fractional depth range of fault activity
    fault_type   : 'normal' (hanging wall down) | 'reverse' (hanging wall up)
    \"\"\"
    NZ, NX = Vp.shape
    x0     = int(x_fault_frac * NX)
    sign   = +1 if fault_type == 'normal' else -1

    Vp_new  = Vp .copy()
    Rho_new = Rho.copy()

    z1 = int(z_top_frac * NZ)
    z2 = int(z_bot_frac * NZ)

    for iz in range(NZ):
        # Dip shifts the fault plane position laterally
        dip_shift = int(iz * np.tan(np.radians(dip_deg)))
        xf = x0 + dip_shift                 # fault plane x at this depth

        # Throw profile: sinusoidal taper
        if z1 <= iz <= z2:
            frac  = (iz - z1) / max(z2 - z1, 1)
            throw = int(sign * throw_max * np.sin(np.pi * frac))
        else:
            throw = 0

        if throw == 0 or xf >= NX or xf < 0:
            continue

        # Hanging-wall: columns to the right of the fault
        row_vp  = Vp [iz, :].copy()
        row_rho = Rho[iz, :].copy()

        # Shift hanging-wall (right of fault) vertically is done column-wise;
        # here we shift the entire row index in depth per column — simpler: roll column
        for ix in range(xf, NX):
            src = iz - throw
            src = np.clip(src, 0, NZ-1)
            Vp_new [iz, ix] = Vp [src, ix]
            Rho_new[iz, ix] = Rho[src, ix]

    return Vp_new, Rho_new

# ── Normal fault at x≈55%, 5° dip, 22-sample throw ───────────
Vp, Rho = apply_fault(Vp, Rho, x_fault_frac=0.55, dip_deg=5,
                      throw_max=22, z_top_frac=0.0, z_bot_frac=0.85,
                      fault_type='normal')

# ── Reverse fault at x≈20%, 8° dip, 15-sample throw ──────────
Vp, Rho = apply_fault(Vp, Rho, x_fault_frac=0.20, dip_deg=8,
                      throw_max=15, z_top_frac=0.1, z_bot_frac=0.90,
                      fault_type='reverse')

print("✅  Normal fault (x≈55%) and reverse fault (x≈20%) added.")
"""))

cells.append(new_markdown_cell("""### 4d  Fracture Zones

Fracture zones act as low-velocity, low-density corridors oriented at arbitrary angles.
They are modelled as tabular bodies with Gaussian-blurred edges to mimic gradational
transitions and introduce chaotic seismic character.
"""))
cells.append(new_code_cell("""def add_fracture_zone(Vp, Rho, x_frac, width_frac, dip_deg,
                      z_top_frac=0.0, z_bot_frac=1.0,
                      vp_factor=0.75, rho_factor=0.90):
    \"\"\"
    Add a dipping fracture zone.
    vp_factor  : multiply existing Vp by this (< 1 = slower = fractured)
    rho_factor : multiply existing Rho
    \"\"\"
    NZ, NX = Vp.shape
    x0     = int(x_frac * NX)
    hw     = int(width_frac * NX / 2)
    z1     = int(z_top_frac * NZ)
    z2     = int(z_bot_frac * NZ)

    mask = np.zeros((NZ, NX), dtype=float)
    for iz in range(z1, z2):
        dip_offset = int(iz * np.tan(np.radians(dip_deg)))
        xc = x0 + dip_offset
        x_lo = max(xc - hw, 0)
        x_hi = min(xc + hw, NX)
        mask[iz, x_lo:x_hi] = 1.0

    # Smooth edges
    mask = gaussian_filter(mask, sigma=2)

    Vp  *= (1 - mask * (1 - vp_factor ))
    Rho *= (1 - mask * (1 - rho_factor))

# ── Two fracture zones ────────────────────────────────────────
add_fracture_zone(Vp, Rho, x_frac=0.65, width_frac=0.025, dip_deg=-12,
                  z_top_frac=0.05, z_bot_frac=0.70)
add_fracture_zone(Vp, Rho, x_frac=0.30, width_frac=0.02,  dip_deg=15,
                  z_top_frac=0.10, z_bot_frac=0.55)

print("✅  Two fracture zones added (x≈65%, x≈30%).")
"""))

cells.append(new_markdown_cell("""### 4e  Unconformity

An **angular unconformity** is a surface of erosion separating older, tilted strata
from younger, flat-lying strata above. We implement it as a depth-varying erosion
surface $z_{\\text{unc}}(x)$ below which older layers are truncated.
"""))
cells.append(new_code_cell("""def add_unconformity(Vp, Rho, z_centre_frac, relief_frac, slope_frac):
    \"\"\"
    Erosional unconformity: above the surface use surface-fill material;
    below, keep existing geology.  z_centre_frac is mid-section depth.
    relief_frac controls the tilted-surface relief.
    slope_frac  controls how much the surface tilts left-to-right.
    \"\"\"
    NZ, NX = Vp.shape
    zc = int(z_centre_frac * NZ)
    relief = int(relief_frac * NZ)

    for ix in range(NX):
        # Unconformity surface: tilted plane + gentle sine relief
        z_unc = int(zc + slope_frac * (ix - NX//2)
                    + relief * 0.3 * np.sin(2*np.pi * ix / NX))
        z_unc = np.clip(z_unc, 5, NZ-5)
        # Fill above with shallow shale (post-unconformity fill)
        Vp [:z_unc, ix] = LITHO['shale'][0]
        Rho[:z_unc, ix] = LITHO['shale'][1]

# ── Unconformity in the left half of section ──────────────────
# (applied only for ix < 0.45*NX to keep folded region intact)
NZ_, NX_ = Vp.shape
for ix in range(int(0.0*NX_), int(0.45*NX_)):
    zc    = int(0.22 * NZ_)
    slope = 0.08
    z_unc = int(zc + slope * (ix - int(0.05*NX_)))
    z_unc = np.clip(z_unc, 5, NZ_-5)
    Vp [:z_unc, ix] = LITHO['shale'][0]
    Rho[:z_unc, ix] = LITHO['shale'][1]

print("✅  Angular unconformity added (left portion of section).")
"""))

cells.append(new_markdown_cell("""### 4f  Salt Dome

Halite ($\\rho \\approx 2160\\,\\text{kg m}^{-3}$, $V_p \\approx 4480\\,\\text{m s}^{-1}$)
is less dense than surrounding sediments at depth, causing **diapirism**.
Salt has nearly uniform acoustic properties, so seismic reflections inside the dome
are absent — a diagnostic blank zone. Velocity pull-up / push-down occurs beneath.
"""))
cells.append(new_code_cell("""def add_salt_dome(Vp, Rho, x_frac, z_top_frac, width_frac, height_frac):
    \"\"\"
    Elliptical salt body.
    \"\"\"
    NZ, NX = Vp.shape
    xc     = int(x_frac    * NX)
    zt     = int(z_top_frac* NZ)
    a      = int(width_frac * NX / 2)   # semi-axis x
    b      = int(height_frac* NZ / 2)   # semi-axis z
    zc     = zt + b

    for iz in range(NZ):
        for ix in range(NX):
            if ((ix-xc)/a)**2 + ((iz-zc)/b)**2 <= 1.0:
                Vp [iz, ix] = LITHO['salt'][0]
                Rho[iz, ix] = LITHO['salt'][1]

# ── Salt dome at x≈85%, top at z≈30%, W=12%, H=28% ───────────
add_salt_dome(Vp, Rho, x_frac=0.85, z_top_frac=0.30,
              width_frac=0.12, height_frac=0.28)

print("✅  Salt dome added (x≈85%).")
"""))

cells.append(new_markdown_cell("""### 4g  Gas Sand & Pinch-Out (Stratigraphic Trap)

A **bright-spot** gas sand has low impedance relative to brine sand, producing a
strong negative reflection coefficient at its top. A **pinch-out** occurs when
the layer tapers to zero thickness laterally.
"""))
cells.append(new_code_cell("""# ── Gas sand lens (bright spot), z=180–210 m, x=40–65% ─────────
for ix in range(int(0.40*NX), int(0.65*NX)):
    Vp [45:53, ix] = LITHO['gas_sand'][0]
    Rho[45:53, ix] = LITHO['gas_sand'][1]

# ── Brine sand pinch-out: tapers from full thickness at x=10% to zero at x=45% ─
for ix in range(int(0.10*NX), int(0.45*NX)):
    frac  = 1 - (ix - int(0.10*NX)) / (int(0.45*NX) - int(0.10*NX))
    thick = int(frac * 12)
    if thick > 0:
        Vp [80:80+thick, ix] = LITHO['brine_sand'][0]
        Rho[80:80+thick, ix] = LITHO['brine_sand'][1]

print("✅  Gas sand bright spot and brine-sand pinch-out added.")
"""))

# ─────────────────────────────────────────────
# 5. WAVELET & CONVOLUTION
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("## 5  Wavelet Design & Convolution Model"))
cells.append(new_code_cell("""# ═══════════════════════════════════════════════════════════
#  COMPUTE ACOUSTIC IMPEDANCE & REFLECTIVITY
# ═══════════════════════════════════════════════════════════
Z = Vp * Rho          # Acoustic impedance [Pa·s/m]

# Normal-incidence reflectivity at each interface
R = np.zeros_like(Z)
R[:-1, :] = (Z[1:, :] - Z[:-1, :]) / (Z[1:, :] + Z[:-1, :] + 1e-9)

print(f"Impedance range  : {Z.min():.0f} – {Z.max():.0f}  Pa·s/m")
print(f"Reflectivity range: {R.min():.4f} – {R.max():.4f}")

# ═══════════════════════════════════════════════════════════
#  WAVELET DEFINITIONS
# ═══════════════════════════════════════════════════════════
def ricker_wavelet(f0, dt, duration=0.128):
    \"\"\"Zero-phase Ricker wavelet.\"\"\"
    t = np.arange(-duration/2, duration/2, dt)
    pft2 = (np.pi * f0 * t)**2
    w = (1 - 2*pft2) * np.exp(-pft2)
    return t, w

def ormsby_wavelet(f1, f2, f3, f4, dt, duration=0.128):
    \"\"\"Ormsby bandpass wavelet in time domain.\"\"\"
    t   = np.arange(-duration/2, duration/2, dt)
    def S(f):  # sinc-squared helper
        return (np.pi*f*t)**2
    def sinc2(f):
        with np.errstate(divide='ignore', invalid='ignore'):
            s = np.sinc(f * t)**2
        return s
    w = ( ((np.pi*f4)**2 / (np.pi*f4 - np.pi*f3)) * sinc2(f4)
        - ((np.pi*f3)**2 / (np.pi*f4 - np.pi*f3)) * sinc2(f3)
        - ((np.pi*f2)**2 / (np.pi*f2 - np.pi*f1)) * sinc2(f2)
        + ((np.pi*f1)**2 / (np.pi*f2 - np.pi*f1)) * sinc2(f1) )
    return t, w / (np.max(np.abs(w)) + 1e-9)

# Build both wavelets
t_rick, w_rick = ricker_wavelet(f0=40, dt=DT)
t_orm,  w_orm  = ormsby_wavelet( 5, 15, 60, 80, DT)

# ── Quick wavelet plot ────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
fig.suptitle('Source Wavelets — Time & Frequency Domain', fontweight='bold')

for ax, t_w, w_w, label, col in [
        (axes[0,0], t_rick, w_rick, 'Ricker 40 Hz — Time', '#2563eb'),
        (axes[0,1], t_orm,  w_orm,  'Ormsby 5-15-60-80 Hz — Time', '#dc2626'),
]:
    ax.plot(t_w*1000, w_w, color=col, lw=2)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_title(label); ax.set_xlabel('Time [ms]'); ax.set_ylabel('Amplitude')
    ax.xaxis.set_minor_locator(MultipleLocator(5))

for ax, t_w, w_w, label, col in [
        (axes[1,0], t_rick, w_rick, 'Ricker 40 Hz — Spectrum', '#2563eb'),
        (axes[1,1], t_orm,  w_orm,  'Ormsby 5-15-60-80 Hz — Spectrum', '#dc2626'),
]:
    freqs = np.fft.rfftfreq(len(w_w), d=DT)
    spec  = np.abs(np.fft.rfft(w_w))
    ax.plot(freqs, spec/spec.max(), color=col, lw=2)
    ax.set_xlim(0, 150); ax.set_title(label)
    ax.set_xlabel('Frequency [Hz]'); ax.set_ylabel('Normalised Amplitude')
    ax.fill_between(freqs, spec/spec.max(), alpha=0.15, color=col)

plt.tight_layout()
plt.savefig('/tmp/wavelets.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅  Wavelets plotted.")
"""))

# ─────────────────────────────────────────────
# 6. FULL SYNTHETIC SEISMIC SECTION
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("## 6  Full Synthetic Seismic Section"))
cells.append(new_code_cell("""# ═══════════════════════════════════════════════════════════
#  CONVOLVE EACH TRACE WITH RICKER WAVELET
# ═══════════════════════════════════════════════════════════
seismic = np.zeros_like(R)
for ix in range(NX):
    seismic[:, ix] = np.convolve(R[:, ix], w_rick, mode='same')

print(f"Seismic section shape : {seismic.shape}")

# ═══════════════════════════════════════════════════════════
#  VISUALISE  — Impedance | Reflectivity | Seismic
# ═══════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 10), sharey=True)

extent = [0, NX*DX/1000, NZ*DZ/1000, 0]   # [km], top to bottom

# — Acoustic impedance —
im0 = axes[0].imshow(Z/1e6, extent=extent, aspect='auto',
                      cmap='RdYlBu_r', vmin=2, vmax=16)
axes[0].set_title('Acoustic Impedance  $Z = \\\\rho V_p$', pad=10)
axes[0].set_xlabel('Distance [km]')
axes[0].set_ylabel('Depth [km]')
plt.colorbar(im0, ax=axes[0], label='Z  [MPa·s/m]', fraction=0.03)

# — Reflectivity —
clim = np.percentile(np.abs(R), 98)
im1 = axes[1].imshow(R, extent=extent, aspect='auto',
                      cmap='RdBu', vmin=-clim, vmax=clim)
axes[1].set_title('Reflectivity Series  $R(z,x)$', pad=10)
axes[1].set_xlabel('Distance [km]')
plt.colorbar(im1, ax=axes[1], label='R', fraction=0.03)

# — Seismic section —
clim2 = np.percentile(np.abs(seismic), 97)
im2 = axes[2].imshow(seismic, extent=extent, aspect='auto',
                      cmap='gray', vmin=-clim2, vmax=clim2)
axes[2].set_title('Synthetic Seismic  $s = w * r$  (Ricker 40 Hz)', pad=10)
axes[2].set_xlabel('Distance [km]')
plt.colorbar(im2, ax=axes[2], label='Amplitude', fraction=0.03)

for ax in axes:
    ax.xaxis.set_minor_locator(MultipleLocator(0.5))
    ax.yaxis.set_minor_locator(MultipleLocator(0.1))

plt.suptitle('Earth Model → Reflectivity → Synthetic Seismic Section',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/tmp/model_seismic.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅  Three-panel figure saved.")
"""))

# ─────────────────────────────────────────────
# 7. SEISMIC ATTRIBUTES
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("## 7  Seismic Attributes"))
cells.append(new_code_cell("""# ═══════════════════════════════════════════════════════════
#  COMPUTE COMPLEX TRACE ATTRIBUTES
# ═══════════════════════════════════════════════════════════
analytic  = hilbert(seismic, axis=0)                   # analytic signal
envelope  = np.abs(analytic)                            # instantaneous amplitude
inst_phase = np.angle(analytic)                         # instantaneous phase
cos_phase  = np.cos(inst_phase)                         # cosine of phase
# Instantaneous frequency (clipped to avoid noise spikes)
inst_freq  = np.diff(np.unwrap(inst_phase, axis=0), axis=0) / (2*np.pi*DT)
inst_freq  = np.clip(inst_freq, 0, 200)
inst_freq  = np.vstack([inst_freq, inst_freq[-1:, :]])  # pad to NZ

print("Attributes computed: envelope, instantaneous phase, cosine phase, instantaneous frequency.")

# ═══════════════════════════════════════════════════════════
#  PLOT ATTRIBUTES
# ═══════════════════════════════════════════════════════════
attrs = [
    (seismic,    'gray',     'Seismic Amplitude',         None),
    (envelope,   'plasma',   'Envelope (Inst. Amplitude)', None),
    (cos_phase,  'hsv',      'Cosine of Inst. Phase',      (-1,1)),
    (inst_freq,  'jet',      'Instantaneous Frequency [Hz]',(0,120)),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharey=True, sharex=True)
axes = axes.ravel()

for ax, (data, cmap, title, vlims) in zip(axes, attrs):
    if vlims is None:
        c = np.percentile(np.abs(data), 97)
        vmin, vmax = -c if cmap=='gray' else 0, c
    else:
        vmin, vmax = vlims
    im = ax.imshow(data, extent=extent, aspect='auto',
                   cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, pad=8, fontsize=12)
    ax.set_xlabel('Distance [km]')
    ax.set_ylabel('Depth [km]')
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)

plt.suptitle('Seismic Attribute Panel', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/attributes.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅  Attribute panel saved.")
"""))

# ─────────────────────────────────────────────
# 8. NOISE & ACQUISITION EFFECTS
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("""## 8  Noise & Acquisition Effects

Real seismic data contains:
- **Random noise** (ambient ground roll, instrument noise) — additive Gaussian
- **Coherent noise** (multiples, ground roll) — bandlimited, aligned along moveout curves
- **Acquisition footprint** — systematic amplitude variation with receiver geometry
- **Diffractions** from point scatterers (fault tips, buried boulders)

We simulate each of these below.
"""))
cells.append(new_code_cell("""# ═══════════════════════════════════════════════════════════
#  1. ADDITIVE RANDOM NOISE
# ═══════════════════════════════════════════════════════════
SNR   = 8.0   # signal-to-noise ratio
sig_s = np.std(seismic)
noise_rnd = np.random.randn(*seismic.shape) * (sig_s / SNR)

# ═══════════════════════════════════════════════════════════
#  2. COHERENT (CORRELATED) NOISE — bandlimited horizontal stripes
# ═══════════════════════════════════════════════════════════
coherent = np.zeros_like(seismic)
for _ in range(6):
    z_start = np.random.randint(10, NZ-20)
    amplitude = np.random.uniform(0.01, 0.04) * sig_s
    coherent[z_start:z_start+3, :] += amplitude

# Low-pass filter the coherent noise
b, a = butter(4, 0.15, btype='low')
for ix in range(NX):
    coherent[:, ix] = filtfilt(b, a, coherent[:, ix])

# ═══════════════════════════════════════════════════════════
#  3. DIFFRACTIONS from fault tips (point scatterers)
# ═══════════════════════════════════════════════════════════
def add_diffraction(section, x_tip, z_tip, velocity, dx, dz, dt, amplitude=0.3):
    \"\"\"Add a hyperbolic diffraction from a point scatterer.\"\"\"
    NZ, NX = section.shape
    t_tip  = z_tip * dz / velocity   # two-way time at scatterer
    for ix in range(NX):
        dist  = abs(ix - x_tip) * dx
        t_arr = np.sqrt(t_tip**2 + (dist/velocity)**2)  # hyperbola
        iz    = int(t_arr / dt)
        if 0 < iz < NZ - len(w_rick):
            section[iz:iz+len(w_rick), ix] += amplitude * w_rick * sig_s

seismic_noisy = seismic + noise_rnd + coherent

# Add diffractions at fault tips
add_diffraction(seismic_noisy, int(0.55*NX), int(0.70*NZ), 3000, DX, DZ, DT*10, 0.25)
add_diffraction(seismic_noisy, int(0.20*NX), int(0.80*NZ), 3200, DX, DZ, DT*10, 0.20)
add_diffraction(seismic_noisy, int(0.85*NX), int(0.58*NZ), 4400, DX, DZ, DT*10, 0.30)

# ── Compare clean vs noisy ────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 9), sharey=True)
clim = np.percentile(np.abs(seismic_noisy), 97)

ax1.imshow(seismic,       extent=extent, aspect='auto', cmap='gray',
           vmin=-clim, vmax=clim)
ax1.set_title('Clean Synthetic Seismic', fontsize=13, pad=8)
ax1.set_xlabel('Distance [km]'); ax1.set_ylabel('Depth [km]')

ax2.imshow(seismic_noisy, extent=extent, aspect='auto', cmap='gray',
           vmin=-clim, vmax=clim)
ax2.set_title(f'Noisy Seismic  (SNR ≈ {SNR}) + Diffractions', fontsize=13, pad=8)
ax2.set_xlabel('Distance [km]')

plt.suptitle('Clean vs. Noisy Synthetic Section', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/noise_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅  Noise & diffraction effects applied.")
"""))

# ─────────────────────────────────────────────
# 9. COMPOSITE FIGURE
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("## 9  Composite Panel — Publication-Quality Figure"))
cells.append(new_code_cell("""fig = plt.figure(figsize=(22, 18))
gs  = GridSpec(3, 3, figure=fig, hspace=0.38, wspace=0.35)

# ── Panel definitions ─────────────────────────────────────────
panel_data = {
    (0,0): (Z/1e6,        'RdYlBu_r', (2,16),  'Acoustic Impedance  [MPa·s/m]', False),
    (0,1): (R,            'RdBu',      None,    'Reflectivity Series  R(z,x)',    True ),
    (0,2): (seismic,      'gray',      None,    'Synthetic Seismic (Ricker 40Hz)',True ),
    (1,0): (seismic_noisy,'gray',      None,    'Seismic + Noise + Diffractions', True ),
    (1,1): (envelope,     'plasma',    (0,None),'Instantaneous Amplitude (Env.)', False),
    (1,2): (cos_phase,    'hsv',       (-1,1),  'Cosine of Instantaneous Phase',  False),
    (2,0): (inst_freq,    'jet',       (0,120), 'Instantaneous Frequency [Hz]',   False),
    (2,1): (Vp/1000,      'viridis',   (1.5,6), 'P-wave Velocity  [km/s]',        False),
    (2,2): (Rho/1000,     'copper',    (1.3,3), 'Bulk Density  [g/cm³]',          False),
}

axes_dict = {}
for (row, col), (data, cmap, vlims, title, sym) in panel_data.items():
    ax = fig.add_subplot(gs[row, col])
    axes_dict[(row,col)] = ax

    if vlims is None:
        c = np.percentile(np.abs(data), 97)
        vmin, vmax = (-c if sym else 0), c
    elif vlims[0] is None:
        vmax = np.percentile(data, 97)
        vmin = 0
    else:
        vmin, vmax = vlims

    im = ax.imshow(data, extent=extent, aspect='auto',
                   cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10, pad=5)
    ax.set_xlabel('Distance [km]', fontsize=8)
    if col == 0:
        ax.set_ylabel('Depth [km]', fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)

fig.suptitle('Synthetic Seismic Survey — Full Geophysical Panel\n'
             '(Faults · Folds · Fractures · Salt Dome · Unconformity · Gas Sand · Diffractions)',
             fontsize=14, fontweight='bold', y=1.005)

plt.savefig('/tmp/composite_panel.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅  Composite panel saved.")
"""))

# ─────────────────────────────────────────────
# 10. INTERPRETATION OVERLAY
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("## 10  Interpretation Overlay"))
cells.append(new_code_cell("""fig, ax = plt.subplots(figsize=(18, 11))

# Base seismic
clim = np.percentile(np.abs(seismic_noisy), 97)
ax.imshow(seismic_noisy, extent=extent, aspect='auto',
          cmap='gray', vmin=-clim, vmax=clim, alpha=0.92)

# ── Overlay colour-coded interpretation annotations ───────────
kw_line  = dict(linewidth=2.5, transform=ax.transData, clip_on=True)
kw_label = dict(fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', alpha=0.7))

x_km = x_pos / 1000
d_km = depth  / 1000

# 1 — Anticline crest (arched arrow)
ax.annotate('', xy=(0.38*NX*DX/1000, 0.08),
            xytext=(0.38*NX*DX/1000, 0.20),
            arrowprops=dict(arrowstyle='->', color='yellow', lw=2))
ax.text(0.38*NX*DX/1000, 0.22, 'Anticline\nCrest',
        color='yellow', ha='center', **kw_label, backgroundcolor='#00000080')

# 2 — Syncline
ax.annotate('', xy=(0.72*NX*DX/1000, 0.30),
            xytext=(0.72*NX*DX/1000, 0.18),
            arrowprops=dict(arrowstyle='->', color='cyan', lw=2))
ax.text(0.72*NX*DX/1000, 0.15, 'Syncline',
        color='cyan', ha='center', **kw_label, backgroundcolor='#00000080')

# 3 — Normal fault (dipping line)
x_f1 = [0.55*NX*DX/1000, 0.57*NX*DX/1000]
z_f1 = [0.0,             0.85*NZ*DZ/1000]
ax.plot(x_f1, z_f1, color='red', **kw_line, linestyle='--')
ax.text(0.575*NX*DX/1000, 0.5*NZ*DZ/1000, 'Normal\nFault',
        color='red', ha='left', **kw_label, backgroundcolor='#00000080')

# 4 — Reverse fault
x_f2 = [0.20*NX*DX/1000, 0.22*NX*DX/1000]
z_f2 = [0.1*NZ*DZ/1000,  0.90*NZ*DZ/1000]
ax.plot(x_f2, z_f2, color='orange', **kw_line, linestyle='-.')
ax.text(0.175*NX*DX/1000, 0.5*NZ*DZ/1000, 'Reverse\nFault',
        color='orange', ha='right', **kw_label, backgroundcolor='#00000080')

# 5 — Fracture zones (shaded rectangles)
for xf, label in [(0.65, 'Fracture\nZone 1'), (0.30, 'Fracture\nZone 2')]:
    rect = mpatches.FancyArrowPatch(
        (xf*NX*DX/1000 - 0.04, 0.05),
        (xf*NX*DX/1000 + 0.04, 0.05),
        color='magenta', arrowstyle='-', linewidth=12, alpha=0.35)
    ax.add_patch(rect)
    ax.axvline(xf*NX*DX/1000, color='magenta', lw=1.2, ls=':', alpha=0.7)
    ax.text(xf*NX*DX/1000, 0.60*NZ*DZ/1000, label,
            color='magenta', ha='center', **kw_label, backgroundcolor='#00000080')

# 6 — Unconformity surface
unc_x = np.linspace(0, 0.45*NX*DX/1000, 80)
slope = 0.08
unc_z = (0.22*NZ*DZ/1000 + slope*(unc_x/DX*1000 - 0.05*NX)*DZ/1000
         + 0.3*0.15*NZ*DZ/1000 * np.sin(2*np.pi * unc_x/(NX*DX/1000)))
ax.plot(unc_x, unc_z, color='lime', lw=2.5, ls='-')
ax.text(0.22*NX*DX/1000, unc_z[40]+0.04, 'Angular\nUnconformity',
        color='lime', ha='center', **kw_label, backgroundcolor='#00000080')

# 7 — Salt dome ellipse
from matplotlib.patches import Ellipse
ell = Ellipse((0.85*NX*DX/1000, (0.30+0.14)*NZ*DZ/1000),
               width=0.12*NX*DX/1000, height=0.28*NZ*DZ/1000,
               edgecolor='deepskyblue', facecolor='none', lw=2.5, ls='-')
ax.add_patch(ell)
ax.text(0.85*NX*DX/1000, (0.30+0.14)*NZ*DZ/1000, 'Salt\nDome',
        color='deepskyblue', ha='center', va='center',
        fontsize=9, fontweight='bold')

# 8 — Gas sand bright spot
ax.annotate('Gas Sand\n(Bright Spot)', xy=(0.52*NX*DX/1000, 45*DZ/1000),
            xytext=(0.52*NX*DX/1000, 0.35),
            arrowprops=dict(arrowstyle='->', color='gold', lw=1.5),
            color='gold', ha='center', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round', alpha=0.7, facecolor='#00000080'))

# 9 — Diffractions labels
for xd, zd, txt in [(0.55, 0.70, 'Diffraction\n(Fault Tip)'),
                    (0.85, 0.58, 'Diffraction\n(Salt Edge)')]:
    ax.text(xd*NX*DX/1000, zd*NZ*DZ/1000+0.1, txt,
            color='wheat', ha='center', fontsize=8.5, fontweight='bold',
            bbox=dict(boxstyle='round', alpha=0.6, facecolor='#00000080'))

ax.set_xlabel('Distance [km]', fontsize=12)
ax.set_ylabel('Depth [km]', fontsize=12)
ax.set_title('Synthetic Seismic Section — Interpreted Geological Features',
             fontsize=14, fontweight='bold', pad=12)
ax.invert_yaxis()
ax.set_ylim([NZ*DZ/1000, 0])

plt.tight_layout()
plt.savefig('/tmp/interpreted_section.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅  Interpreted section saved.")
"""))

# ─────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────
cells.append(new_markdown_cell("""---

## 📋 Summary

| Feature | Location | Method |
|---------|----------|--------|
| Horizontal layers (10+) | Full section | Direct array assignment |
| **Anticline** | x ≈ 38% | Gaussian-modulated column roll |
| **Syncline** | x ≈ 72% | Gaussian-modulated column roll |
| **Normal Fault** | x ≈ 55% | Variable-throw row displacement |
| **Reverse Fault** | x ≈ 20% | Variable-throw row displacement |
| **Fracture Zone 1** | x ≈ 65%, dip 12° | Gaussian-blurred low-Vp corridor |
| **Fracture Zone 2** | x ≈ 30%, dip 15° | Gaussian-blurred low-Vp corridor |
| **Unconformity** | x = 0–45%, z ≈ 22% | Tilted erosion surface |
| **Salt Dome** | x ≈ 85% | Elliptical halite body |
| **Gas Sand (Bright Spot)** | x = 40–65%, z ≈ 180 m | Low-impedance lens |
| **Brine Sand Pinch-out** | x = 10–45% | Tapering layer |
| **Diffractions** | Fault tips + salt edge | Hyperbolic forward modelling |
| **Random + Coherent Noise** | Full section | Gaussian + bandpass |

### Key Equations Recap

$$R_i = \\frac{Z_{i+1}-Z_i}{Z_{i+1}+Z_i}, \\quad Z=\\rho V_p, \\quad s(t)=w(t)*r(t)+n(t)$$

### Suggested Extensions

- Convert depth → two-way time using velocity integration: $t(z) = 2\\int_0^z \\frac{dz'}{V_p(z')}$
- Add **amplitude-versus-offset (AVO)** modelling using Zoeppritz equations  
- Model **multiple reflections** (peg-leg, interbed)  
- Apply **migration** to collapse diffractions  
- Add **anisotropy** (VTI/HTI) for fractured reservoir modelling  

---

*Generated with NumPy, SciPy, and Matplotlib — no proprietary seismic packages required.*
"""))

# ─────────────────────────────────────────────
# WRITE NOTEBOOK
# ─────────────────────────────────────────────
nb = new_notebook(cells=cells)
nb.metadata.update({
    "kernelspec": {
        "display_name": "Python 3",
        "language": "python",
        "name": "python3"
    },
    "language_info": {
        "name": "python",
        "version": "3.10.0"
    }
})

out_path = 'seismic_output/synthetic_seismic_survey.ipynb'
with open(out_path, 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

print(f"✅  Notebook written → {out_path}")

IsADirectoryError: [Errno 21] Is a directory: 'seismic_output'

In [2]:
pip install nbformat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 KB 31.6 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 KB 616.8 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.5/390.5 KB 142.0 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.5/67.5 KB 86.8 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.
